# Domain 1: Agentic Architecture & Orchestration
## TS 1.7: Manage Session State, Resumption, and Forking
**Certification Focus**: CCAR-F Domain 1 (27% Exam Weight) | **Scenario Alignment**: Scenario 2 & 4

---

### 🎯 Key Architectural Principles
1. **Session Resumption**: `--resume <session-id>` reloads historical message trails for continuous sequential interactions.
2. **Session Forking (`fork_session`)**: Creates an isolated branch from a common parent baseline. Mutations in child branches do **not** pollute the parent session or sibling branches.
3. **The Stale Context Trap**: Resuming long historical sessions after the underlying codebase has changed causes Claude to hallucinate based on cached conversation snippets rather than reading the updated files.
4. **Remedy for Stale Context**: Discard the bloated conversation history and inject a concise, structured Markdown summary into a fresh session.

In [ ]:
# Step 1: Environment & Offline Setup
import copy
import sys
import uuid
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any, Dict, List, Optional

repo_root = Path.cwd()
while repo_root != repo_root.parent and not (repo_root / "pyproject.toml").exists():
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

print(" Environment initialized for TS 1.7 Session Management.")

### 🗃️ Step 2: Session State & Manager Implementation
We implement `SessionState` supporting branching/forking and `SessionManager` supporting summary injection.

In [ ]:
# Step 2: Session Manager Logic
@dataclass
class SessionState:
    session_id: str
    session_name: str
    messages: List[Dict[str, Any]] = field(default_factory=list)
    metadata: Dict[str, Any] = field(default_factory=dict)

    def fork(self, new_name: str) -> "SessionState":
        """Branch from current session checkpoint into an isolated thread."""
        return SessionState(
            session_id=f"fork_{uuid.uuid4().hex[:8]}",
            session_name=new_name,
            messages=copy.deepcopy(self.messages),
            metadata={"parent_session_id": self.session_id},
        )


class SessionManager:
    def __init__(self) -> None:
        self.sessions: Dict[str, SessionState] = {}

    def create_session(self, name: str, initial_messages: Optional[List[Dict[str, Any]]] = None) -> SessionState:
        sid = f"sess_{uuid.uuid4().hex[:8]}"
        sess = SessionState(session_id=sid, session_name=name, messages=initial_messages or [])
        self.sessions[name] = sess
        return sess

    def resume_with_injected_summary(self, name: str, structured_summary: str, new_instruction: str) -> SessionState:
        """
        Stale Context Remedy: Discards bloated history and starts a clean session
        anchored with a structured summary of previous findings.
        """
        clean_messages = [
            {"role": "user", "content": f"### PREVIOUS ARCHITECTURAL CONTEXT\n{structured_summary}"},
            {"role": "assistant", "content": "Context acknowledged. Ready for next instructions."},
            {"role": "user", "content": new_instruction},
        ]
        return self.create_session(f"{name}_resumed", clean_messages)

print(" SessionManager declared with forking and summary injection.")

### 🧪 Step 3: Interactive Simulation
1. Create a base session.
2. Fork the session to test an alternative architecture in `branch_a`.
3. Verify that adding messages to `branch_a` leaves `base` unpolluted.
4. Mitigate stale context by resuming with an injected summary.

In [ ]:
# Step 3: Simulation
mgr = SessionManager()

# 1. Base session
base = mgr.create_session("base", [{"role": "user", "content": "Design database schema."}])
print(f"Created base session: {base.session_name} with {len(base.messages)} message(s)")

# 2. Fork into branch_a
branch_a = base.fork("branch_a")
branch_a.messages.append({"role": "user", "content": "Explore NoSQL option (MongoDB)"})
print(f"Forked {branch_a.session_name} with {len(branch_a.messages)} messages")
print(f"Base session message count remains: {len(base.messages)}")

# 3. Resume with injected summary
clean = mgr.resume_with_injected_summary(
    "base",
    structured_summary="Agreed schema: PostgreSQL with orders and items tables.",
    new_instruction="Add index on customer_id.",
)
print(f"\nClean resumed session created: {clean.session_name}")
print("Clean messages count:", len(clean.messages))

### 🔬 Step 4: Verification Assertions
We assert that:
1. Forked session preserves parent history at fork time.
2. Mutating the child session does not affect the parent.
3. Resume with summary compresses context and clears stale messages.

In [ ]:
# Step 4: Verification Checks
assert branch_a.metadata["parent_session_id"] == base.session_id
assert len(branch_a.messages) == 2
assert len(base.messages) == 1, "Parent session must not be polluted by child fork!"

# Verify summary injection
assert "PREVIOUS ARCHITECTURAL CONTEXT" in clean.messages[0]["content"]
assert "Add index on customer_id" in clean.messages[2]["content"]

print(" All verification assertions passed! Session management verified.")

### 💡 CCAR-F Exam Golden Rules
- **Rule 1**: Use `fork_session` for divergent exploration paths; never mutate the primary branch.
- **Rule 2**: When files or requirements change, discard long sessions and inject a **structured Markdown summary** to defeat the stale context trap.
- **Rule 3**: Do not use `--resume` indefinitely on large refactors; stale caches cause hallucinations.